# GDBN v5: Point-in-Time, Walk-Forward Volatility

Notebook 04 found that the GDBN + HAR **beat HAR** at predicting next-day volatility (fixed 30 stocks, one training period). Does that hold **without hindsight**?

- **Universe:** the same point-in-time snapshots as notebook 06 (5 largest S&P 500 stocks per sector each quarter, about 50–55).
- **Walk-forward:** each quarter, re-learn on only the **past 3 years**, then predict every day of the next quarter (out of sample).
- **Volatility:** $\text{vol}_i(t) = \log(|r_i(t)| + 0.0001)$, as in notebook 04.
- **Nodes (targets):** each stock's volatility today.
- **Inputs at $t-1$:** every stock's volatility, sentiment, and attention, the 5 ETFs' volatility, plus **HAR terms** (past-week and past-month average volatility) for every stock and ETF.
- **Benchmark: HAR** for each stock (its own volatility yesterday, past week, past month), fit on the same window.

**Two ways to choose λ** (both use only past data):

| Version | λ |
|---|---|
| **λ by MSE, per stock** | inside each window: fit on the first 2 years, pick the λ with the lowest error on the last year, use it with the full 3-year fit |
| **λ shared** | one λ for all stocks, the one with the lowest error over the 2006–2020 walk-forward |

**Periods:** 2006–2020 walk-forward (choose the shared λ) and 2021–2022 validation. Test years (2023–2025) are **not used**.

**Survivorship check:** the same walk-forward on the fixed 30 stocks. Takes several minutes to run.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import lasso_path
from gdbn_utils import DATA, RESULTS, dm_report, Z90, VOL_FLOOR

BUILT = DATA / "pit" / "built"
WINDOW, INNER = 756, 252            # 3-year training window; last year = inner validation for λ by MSE
SENT_KIND = "news"
ALPHAS = np.logspace(-0.7, -3, 20)
TUNE_END, EVAL_END = "2020-12-31", "2022-12-31"   # test years locked
ETFS = ["SPY", "USO", "GLD", "DBC", "TLT"]

days = pd.DatetimeIndex(pd.read_csv(BUILT / "trading_days.csv", parse_dates=["date"])["date"])
panel = pd.read_csv(BUILT / "panel.csv.gz", parse_dates=["date"],
                    usecols=["date", "permno", "ret", f"n_{SENT_KIND}", f"sent_{SENT_KIND}"])
RET = panel.pivot(index="date", columns="permno", values="ret").reindex(days)
VOL = np.log(RET.abs() + VOL_FLOOR)
SENT = panel.pivot(index="date", columns="permno", values=f"sent_{SENT_KIND}").reindex(days)
ATTN = np.log1p(panel.pivot(index="date", columns="permno", values=f"n_{SENT_KIND}")).reindex(days)
ETFV = np.log(pd.read_csv(BUILT / "etf_returns.csv", index_col="date", parse_dates=True).reindex(days)[ETFS].abs() + VOL_FLOOR)
del panel

universe = pd.read_csv(BUILT / "universe.csv", parse_dates=["rebalance_date"])
sel = universe[universe["selected"]]
rebal = [pd.Timestamp(r) for r in sorted(sel["rebalance_date"].unique()) if r <= pd.Timestamp(EVAL_END)]
U_PIT = {r: list(g["permno"]) for r, g in sel.groupby("rebalance_date") if r in rebal}
SECTOR = {(r, p): s for r, p, s in zip(sel["rebalance_date"], sel["permno"], sel["sector_name"])}
TICKER = sel.sort_values("rebalance_date").drop_duplicates("permno", keep="last").set_index("permno")["ticker"].to_dict()
fixed = pd.read_csv(DATA / "universe.csv").drop_duplicates("ticker")
fixed = fixed[fixed["rp_entity_id"].notna()]
U_FIXED = {r: [int(p) for p in fixed["permno"]] for r in rebal}
print(f"{len(rebal)} quarters; point-in-time and fixed-30 universes")

## 1. Walk-forward: one lasso per stock each quarter, with HAR as the benchmark

For each quarter and each stock in the universe:
- **GDBN + HAR (lasso):** predicts the stock's volatility from all inputs at $t-1$ (z-scored with the window's statistics). Predictions are **levels** (they include the stock's average), since volatility levels matter.
- **HAR:** ordinary regression on the stock's own volatility yesterday, past week, past month.
- **σ** for 90% ranges: each model's training residual standard deviation.

In [ ]:
def fit_quarter(r, nxt, U):
    pos = days.searchsorted(r)
    if pos - WINDOW - 23 < 0:
        return None
    hold = days[pos: days.searchsorted(nxt)]
    hold = hold[hold <= pd.Timestamp(EVAL_END)]
    if len(hold) == 0:
        return None
    win = days[pos - WINDOW: pos]
    span = days[pos - WINDOW - 23: pos + len(hold)]

    V = pd.concat([VOL.reindex(index=span, columns=U), ETFV.reindex(span)], axis=1)          # stocks + ETF volatility
    V.columns = [str(c) for c in V.columns]
    week, month = V.rolling(5, min_periods=3).mean(), V.rolling(22, min_periods=10).mean()
    X = pd.concat([V.add_suffix("|vol"), SENT.reindex(index=span, columns=U).add_suffix("|sent"),
                   ATTN.reindex(index=span, columns=U).add_suffix("|attn"),
                   week.add_suffix("|week"), month.add_suffix("|month")], axis=1).shift(1)
    Xw, Xh = X.loc[win], X.loc[hold]
    mu, sd = Xw.mean(), Xw.std().replace(0, np.nan)
    Xwz, Xhz = ((Xw - mu) / sd).fillna(0).values, ((Xh - mu) / sd).fillna(0).values
    xm = Xwz.mean(axis=0)

    n = len(U)
    P = np.full((len(ALPHAS), len(hold), n), np.nan)       # GDBN predictions for every λ
    SIG = np.full((len(ALPHAS), n), np.nan)
    W = np.zeros((n, X.shape[1], len(ALPHAS)))
    har, har_sig = np.full((len(hold), n), np.nan), np.full(n, np.nan)
    k_mse = np.full(n, -1)
    for i, p in enumerate(U):
        y = V.loc[win, str(p)].values
        ok = ~np.isnan(y)
        if ok.sum() < 504:
            continue
        ym, ys = y[ok].mean(), y[ok].std()
        yz = (y[ok] - ym) / ys
        _, coefs, _ = lasso_path(Xwz[ok] - xm, yz, alphas=ALPHAS)
        W[i] = coefs
        P[:, :, i] = (ym + ((Xhz - xm) @ coefs) * ys).T
        SIG[:, i] = ((yz[:, None] - (Xwz[ok] - xm) @ coefs).std(axis=0)) * ys

        rows_ok = np.where(ok)[0]                           # λ by MSE: inner validation = last year
        tr_i, va_i = rows_ok[:-INNER], rows_ok[-INNER:]
        yt, yv = (y[tr_i] - ym) / ys, (y[va_i] - ym) / ys
        xmi = Xwz[tr_i].mean(axis=0)
        _, c_in, _ = lasso_path(Xwz[tr_i] - xmi, yt - yt.mean(), alphas=ALPHAS)
        k_mse[i] = int(np.argmin(((yv[:, None] - (yt.mean() + (Xwz[va_i] - xmi) @ c_in)) ** 2).mean(axis=0)))

        F = X[[f"{p}|vol", f"{p}|week", f"{p}|month"]]       # HAR benchmark (already lagged)
        fw = F.loc[win]
        okh = fw.notna().all(axis=1).values & ok
        A = np.column_stack([np.ones(okh.sum()), fw.values[okh]])
        b, *_ = np.linalg.lstsq(A, y[okh], rcond=None)
        har[:, i] = b[0] + F.loc[hold].fillna(fw[okh].mean()).values @ b[1:]
        har_sig[i] = (y[okh] - A @ b).std(ddof=4)
    actual = V.loc[hold, [str(p) for p in U]].values
    return {"r": r, "hold": hold, "U": U, "P": P, "SIG": SIG, "W": W, "har": har, "har_sig": har_sig,
            "k_mse": k_mse, "actual": actual, "cols": list(X.columns)}


def walk_forward(U_by_q, label):
    out = []
    for j, r in enumerate(rebal):
        nxt = rebal[j + 1] if j + 1 < len(rebal) else pd.Timestamp(EVAL_END) + pd.Timedelta(days=1)
        res = fit_quarter(r, pd.Timestamp(nxt), U_by_q[r])
        if res is not None:
            out.append(res)
        if (j + 1) % 20 == 0:
            print(f"  {label}: {j + 1} of {len(rebal)} quarters")
    return out


pit = walk_forward(U_PIT, "point-in-time")
fix = walk_forward(U_FIXED, "fixed 30")

## 2. Choose the shared λ on the 2006–2020 walk-forward

Lowest total squared error over all out-of-sample stock-days in 2006–2020.

In [ ]:
def frame(results, what, k=None):
    # date x permno table of predictions ("gdbn" at λ index k, "mse" = per-stock λ, "har") or actual volatility
    parts = []
    for q in results:
        if what == "gdbn":
            v = q["P"][k]
        elif what == "mse":
            v = np.full(q["har"].shape, np.nan)
            for i, kk in enumerate(q["k_mse"]):
                if kk >= 0:
                    v[:, i] = q["P"][kk, :, i]
        else:
            v = q[what]
        parts.append(pd.DataFrame(v, index=q["hold"], columns=q["U"]))
    return pd.concat(parts).sort_index()


def sigma_frame(results, what, k=None):
    parts = []
    for q in results:
        if what == "gdbn":
            s = q["SIG"][k]
        elif what == "mse":
            s = np.array([q["SIG"][kk, i] if kk >= 0 else np.nan for i, kk in enumerate(q["k_mse"])])
        else:
            s = q["har_sig"]
        parts.append(pd.DataFrame(np.tile(s, (len(q["hold"]), 1)), index=q["hold"], columns=q["U"]))
    return pd.concat(parts).sort_index()


def mse_curve(results):
    act, har = frame(results, "actual").loc[:TUNE_END], frame(results, "har").loc[:TUNE_END]
    ok = act.notna() & har.notna()
    return np.array([((act - frame(results, "gdbn", k).loc[:TUNE_END]) ** 2).where(ok).stack().mean()
                     for k in range(len(ALPHAS))]), ((act - har) ** 2).where(ok).stack().mean()

curve_pit, har_pit = mse_curve(pit)
curve_fix, har_fix = mse_curve(fix)
k_pit, k_fix = int(np.argmin(curve_pit)), int(np.argmin(curve_fix))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.semilogx(ALPHAS, 100 * (1 - curve_pit / har_pit), "o-", ms=3, label="point-in-time")
ax.semilogx(ALPHAS, 100 * (1 - curve_fix / har_fix), "s-", ms=3, label="fixed 30 stocks")
ax.axhline(0, color="black", lw=0.8); ax.invert_xaxis()
ax.set_xlabel("λ  (left: few arrows, right: many arrows)"); ax.set_ylabel("R² vs HAR (%), 2006–2020")
ax.set_title("Walk-forward accuracy vs HAR along the λ grid (above 0 = beats HAR)"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Shared λ: point-in-time {ALPHAS[k_pit]:.4f}, fixed 30 {ALPHAS[k_fix]:.4f}")

## 3. Results vs HAR

- **R² vs HAR:** $1 - \text{MSE}_{model} / \text{MSE}_{HAR}$ over all stock-days. **Positive = beats HAR.**
- **Diebold–Mariano test** vs HAR (each day's average loss across stocks, Newey–West).
- **Stocks beating HAR:** share of stocks (with at least 60 days in the period) where the model's MSE is lower.
- **coverage90:** share of days the actual volatility falls inside prediction ± 1.645 σ (should be about 90%).

In [ ]:
def evaluate(results, k, start, end):
    act = frame(results, "actual").loc[start:end]
    preds = {"HAR": (frame(results, "har"), sigma_frame(results, "har")),
             "GDBN + HAR (λ shared)": (frame(results, "gdbn", k), sigma_frame(results, "gdbn", k)),
             "GDBN + HAR (λ by MSE, per stock)": (frame(results, "mse"), sigma_frame(results, "mse"))}
    ok = act.notna()
    for p, _ in preds.values():
        ok &= p.loc[start:end].notna()
    err = {n: (act - p.loc[start:end]).where(ok) for n, (p, _) in preds.items()}
    har_mse = (err["HAR"] ** 2).stack().mean()
    rows, lines = {}, []
    for n, (p, s) in preds.items():
        e = err[n]
        per_stock = (e ** 2).mean() / (err["HAR"] ** 2).mean()
        enough = e.notna().sum() >= 60
        rows[n] = {"R² vs HAR (%)": 100 * (1 - (e ** 2).stack().mean() / har_mse),
                   "stocks beating HAR": (per_stock[enough] < 1).mean() if n != "HAR" else np.nan,
                   "coverage90": (e.abs() <= Z90 * s.loc[start:end]).where(ok).stack().mean()}
        if n != "HAR":
            lines.append(dm_report(n[:22], "HAR", e, err["HAR"]))
    return pd.DataFrame(rows).T, lines


tables = {}
for name, (start, end) in {"2006–2020 walk-forward": (None, TUNE_END), "2021–2022 validation": ("2021-01-01", EVAL_END)}.items():
    for label, res, k in [("point-in-time", pit, k_pit), ("fixed 30 (hindsight)", fix, k_fix)]:
        tab, lines = evaluate(res, k, start, end)
        tables[(name, label)] = tab
        print(f"===== {name}: {label} =====")
        print(tab.round(3).to_string())
        print("\n".join("  " + l for l in lines), "\n")

In [ ]:
# R² vs HAR over time (each quarter), point-in-time
act, har = frame(pit, "actual"), frame(pit, "har")
gd, ms = frame(pit, "gdbn", k_pit), frame(pit, "mse")
ok = act.notna() & har.notna() & gd.notna() & ms.notna()
q = act.index.to_period("Q")
def r2_by_q(pred):
    num = ((act - pred) ** 2).where(ok).sum(axis=1).groupby(q).sum()
    den = ((act - har) ** 2).where(ok).sum(axis=1).groupby(q).sum()
    return 100 * (1 - num / den)
fig, ax = plt.subplots(figsize=(12, 3.5))
r2_by_q(gd).plot(ax=ax, label="λ shared"); r2_by_q(ms).plot(ax=ax, label="λ by MSE, per stock")
ax.axhline(0, color="black", lw=0.8); ax.set_ylabel("R² vs HAR (%)")
ax.set_title("Point-in-time GDBN vs HAR, each quarter (above 0 = beats HAR)"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Quarters where the GDBN (λ shared) beats HAR: {(r2_by_q(gd) > 0).mean():.0%}")

## 4. Volatility spillover network over time (point-in-time, shared λ)

Arrows from one stock's volatility (yesterday, week, or month average) to another stock's volatility today. Same-sector share vs chance, and how many arrows persist from one quarter to the next.

In [ ]:
rows, prev = [], None
for qr in pit:
    Wk = qr["W"][:, :, k_pit]
    i_idx, j_idx = np.nonzero(Wk)
    src = [qr["cols"][j] for j in j_idx]
    tgt = [qr["U"][i] for i in i_idx]
    E = pd.DataFrame({"source": src, "target": tgt, "weight": Wk[i_idx, j_idx]})
    E["src_id"], E["src_kind"] = E["source"].str.split("|").str[0], E["source"].str.split("|").str[1]
    vv = E[E["src_kind"].isin(["vol", "week", "month"]) & E["src_id"].str.isdigit()]
    vv = vv[vv["src_id"].astype(int) != vv["target"]]
    same = np.mean([SECTOR.get((qr["r"], int(a))) == SECTOR.get((qr["r"], b)) for a, b in zip(vv["src_id"], vv["target"])]) if len(vv) else np.nan
    chance = np.mean([SECTOR.get((qr["r"], a)) == SECTOR.get((qr["r"], b)) for a in qr["U"] for b in qr["U"] if a != b])
    cur = set(zip(vv["src_id"].astype(int), vv["target"]))
    pers = np.nan
    if prev is not None:
        both = set(qr["U"]) & prev[1]
        a = {e for e in cur if e[0] in both and e[1] in both}
        b = {e for e in prev[0] if e[0] in both and e[1] in both}
        pers = len(a & b) / len(a | b) if (a | b) else np.nan
    news = E[E["src_kind"].isin(["sent", "attn"])]
    rows.append({"quarter": qr["r"], "arrows": len(E), "vol spillovers": len(vv), "same-sector share": same,
                 "chance": chance, "persistence": pers, "news arrows": len(news)})
    prev = (cur, set(qr["U"]))
net = pd.DataFrame(rows).set_index("quarter")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
net[["vol spillovers", "news arrows"]].plot(ax=axes[0], title="Arrows per quarter")
net[["same-sector share", "chance"]].rolling(4).mean().plot(ax=axes[1], title="Spillovers within a sector (1-year avg)")
net["persistence"].rolling(4).mean().plot(ax=axes[2], title="Spillovers kept from the previous quarter (1-year avg)")
for ax in axes:
    ax.axvline(pd.Timestamp("2008-09-15"), color="red", ls=":", lw=1); ax.axvline(pd.Timestamp("2020-03-01"), color="red", ls=":", lw=1)
plt.tight_layout(); plt.show()
print(net.describe().loc[["mean", "min", "max"]].round(3).to_string())

## 5. Save results

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
pd.concat(tables, names=["period", "universe"]).to_csv(RESULTS / "pit_vol_results.csv")
net.to_csv(RESULTS / "pit_vol_network_stats.csv")
pd.Series({"alpha_pit": ALPHAS[k_pit], "alpha_fixed": ALPHAS[k_fix], "window": WINDOW, "inner": INNER}).to_csv(RESULTS / "pit_vol_settings.csv")
print("Saved to", RESULTS)